# Shared Paper 2 split and model preprocessing

Freeze the existing 50/50 article split and create one common feature contract for all ranking models. Shared transformations produce author reception rates, discussion pace, prior reply composition, and leakage-safe centred reply depth. Recent-activity share and branch activity counts are excluded from the primary contract.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display

from commentgap_analysis.preprocessing import prepare_shared_model_data

SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
LEGACY_SPLIT = Path(os.getenv("COMMENTGAP_FROZEN_SPLIT", "model_output/selection_2025/xgboost_paper2/master_article_split.parquet"))
FORCE = os.getenv("COMMENTGAP_PREPROCESS_FORCE", "0").lower() in {"1", "true", "yes"}
assert LEGACY_SPLIT.exists(), f"Frozen Paper 2 split is missing: {LEGACY_SPLIT}"

{"feature_root": str(FEATURE_ROOT), "output_root": str(MODEL_DATA_ROOT), "frozen_split": str(LEGACY_SPLIT), "force": FORCE}

{'feature_root': 'model_output/selection_2025/features',
 'output_root': 'model_output/selection_2025/model_data',
 'frozen_split': 'model_output/selection_2025/xgboost_paper2/master_article_split.parquet',
 'force': True}

In [2]:
result = prepare_shared_model_data(
    FEATURE_ROOT,
    MODEL_DATA_ROOT,
    frozen_split_path=LEGACY_SPLIT,
    seed=SEED,
    development_folds=5,
    force=FORCE,
)
result["rows"]

{'root': 710821, 'all': 2392952}

In [3]:
display(result["article_split"]["split_role"].value_counts().rename("articles"))
display(result["split_balance"])
display(result["reply_depth_centers"])
for scope in ("root", "all"):
    features = result["feature_manifest"]["models"][scope]["features"]
    transformed = {
        "prior_reply_composition", "discussion_pace",
        "reply_depth_centered",
        "author_prior_30d_upvote_reception",
        "author_prior_30d_downvote_reception",
    }
    print(scope, len(features), [name for name in features if name in transformed])

split_role
development    2559
paper2_test    2559
Name: articles, dtype: int64

,covariate,development_mean,paper2_test_mean,standardized_mean_difference,abs_standardized_mean_difference,acceptance_threshold,accepted
0,log1p_n_candidates_root,4.573854,4.572367,0.001707,0.001707,0.05,True
1,log1p_n_candidates_all,5.737229,5.757111,-0.021059,0.021059,0.05,True
2,n_picks_root,3.940211,3.932395,0.004736,0.004736,0.05,True
3,n_picks_all,4.091833,4.087143,0.002777,0.002777,0.05,True
4,reply_proportion,0.669957,0.675579,-0.046128,0.046128,0.05,True


{'full_development': 1.097516362954242,
 'fold_00_training': 1.096525895639284,
 'fold_01_training': 1.0991133691926884,
 'fold_02_training': 1.0972366904824031,
 'fold_03_training': 1.0954266403680566,
 'fold_04_training': 1.0992653585707404}

root 39 ['prior_reply_composition', 'discussion_pace', 'author_prior_30d_upvote_reception', 'author_prior_30d_downvote_reception']
all 41 ['prior_reply_composition', 'discussion_pace', 'author_prior_30d_upvote_reception', 'author_prior_30d_downvote_reception', 'reply_depth_centered']


## Downstream contract

All regression, XGBoost, and neural ranking workflows read the same v4 model-data contract. It contains discussion pace instead of total prior-comment counts, omits recent-activity share and branch activity, and substitutes fold-specific centred reply depth during development CV.